# From Effects to Decisions
## P@S Lecture 18: You estimated the effect. Now what?

Coppock, Hill & Vavreck (2020) ran 59 weekly experiments during the 2016
presidential campaign and found that political ads have small effects.
Today: replicate their key findings with REAL replication data, then
explore what this means for campaign strategy.

Replication data: Harvard Dataverse, DOI 10.7910/DVN/TN7KWR

In [ ]:
# Install pyreadr for reading R data files from the Dataverse
# (run this cell once; restart runtime if needed)
!pip install pyreadr -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pyreadr  # for reading .rds files from Harvard Dataverse
import io
import requests

np.random.seed(42)  # reproducibility for simulation sections later
plt.style.use('seaborn-v0_8-whitegrid')

# Color palette used throughout
BLUE, RED, GREEN, GRAY, ORANGE = '#2980b9', '#c0392b', '#27ae60', '#7f8c8d', '#e67e22'

---
## Section 1: Load real replication data from Harvard Dataverse

Coppock, Hill & Vavreck deposited their replication data at:
https://doi.org/10.7910/DVN/TN7KWR

We download four files:
- **Favorability SATEs** (59 experiments, sample average treatment effects)
- **Vote choice SATEs** (34 experiments)
- **Favorability CATEs** (conditional average treatment effects by subgroup)
- **Ad metadata** (which party each ad supports, ad titles, dates)

In [ ]:
# Harvard Dataverse file IDs for the Coppock et al. replication archive
FILE_IDS = {
    'fav_sate':   3988299,  # favorability SATEs (59 rows)
    'vote_sate':  3988301,  # vote choice SATEs (34 rows)
    'fav_cate':   3988303,  # favorability CATEs (354 rows)
    'vote_cate':  3988296,  # vote choice CATEs (204 rows)
    'ad_meta':    3988290,  # ad metadata (52 ads, tab-separated)
}

DATAVERSE_URL = 'https://dataverse.harvard.edu/api/access/datafile/{}'

def download_rds(file_id):
    """Download an .rds file from Harvard Dataverse and return a DataFrame."""
    url = DATAVERSE_URL.format(file_id)
    resp = requests.get(url)  # fetch the binary .rds content
    resp.raise_for_status()
    # pyreadr.read_r can read from a file path, so we write to a temp buffer
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix='.rds', delete=False) as f:
        f.write(resp.content)
        tmp_path = f.name
    result = pyreadr.read_r(tmp_path)  # returns OrderedDict of DataFrames
    os.unlink(tmp_path)  # clean up temp file
    return list(result.values())[0]  # .rds files contain a single object

def download_tsv(file_id):
    """Download a tab-separated metadata file from Dataverse."""
    url = DATAVERSE_URL.format(file_id)
    resp = requests.get(url)
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.text), sep='\t')

print("Downloading replication data from Harvard Dataverse...")
fav_sate = download_rds(FILE_IDS['fav_sate'])    # favorability ATEs
vote_sate = download_rds(FILE_IDS['vote_sate'])   # vote choice ATEs
fav_cate = download_rds(FILE_IDS['fav_cate'])     # favorability CATEs
vote_cate = download_rds(FILE_IDS['vote_cate'])   # vote choice CATEs
ad_meta = download_tsv(FILE_IDS['ad_meta'])        # ad metadata
print("Done! Loaded 5 files from Dataverse.")

In [ ]:
# Quick look at what we got
print("=== Favorability SATEs ===")
print(f"Shape: {fav_sate.shape}")
print(f"Columns: {list(fav_sate.columns)}")
print(fav_sate.head(3))
print()
print("=== Ad Metadata ===")
print(f"Shape: {ad_meta.shape}")
print(f"Columns: {list(ad_meta.columns)}")
print(ad_meta.head(3))

**What you should see above:** The favorability SATEs file has 59 rows (one per
experiment) with columns for the estimate, standard error, and experiment
identifiers. The ad metadata tells us which party each ad supports, the ad
title, and when it aired.

---
## Section 2: Explore the experiments

Before plotting, let us understand the data. Each row in `fav_sate` is one
experiment. Coppock et al. ran these weekly during the 2016 campaign, testing
real TV ads from both parties on a nationally representative sample.

In [ ]:
# Merge ad metadata onto the SATE results so we know which party each ad supports
# The merge key depends on column names; let's inspect and find the common key
print("fav_sate columns:", list(fav_sate.columns))
print("ad_meta columns:", list(ad_meta.columns))

In [ ]:
# Identify the pro_democrat indicator
# In the Coppock data, pro_democrat == 1 means the ad supports the Democratic candidate
if 'pro_democrat' in fav_sate.columns:
    fav_sate['ad_party'] = fav_sate['pro_democrat'].map({1: 'Democratic', 0: 'Republican'})
elif 'pro_democrat' in ad_meta.columns:
    # Need to merge first; find common column
    common_cols = set(fav_sate.columns) & set(ad_meta.columns)
    print(f"Common columns for merge: {common_cols}")
    # Merge on the common key to get pro_democrat
    merge_key = list(common_cols - {'estimate', 'std.error', 'conf.low', 'conf.high'})[0]
    fav_sate = fav_sate.merge(
        ad_meta[['pro_democrat', merge_key]].drop_duplicates(),
        on=merge_key, how='left'
    )
    fav_sate['ad_party'] = fav_sate['pro_democrat'].map({1: 'Democratic', 0: 'Republican'})

# Count experiments by party
print("\nExperiments by ad party:")
print(fav_sate['ad_party'].value_counts())
print(f"\nTotal experiments: {len(fav_sate)}")

# Show summary statistics for the ATEs
print(f"\nATE summary (favorability scale):")
print(fav_sate['estimate'].describe().round(4))

**What you should see:** About 59 experiments split roughly evenly between
Democratic and Republican ads. The ATEs are small, centered near zero,
with most falling between -0.1 and +0.1 on the favorability scale.

---
## Section 3: Forest plot of real ATEs

This is the key figure from Coppock et al. Each horizontal line is one
experiment. The point is the estimated ATE; the whiskers show the 95%
confidence interval. Blue = pro-Democratic ad, Red = pro-Republican ad.
The diamond at the bottom is the meta-analytic average.

In [ ]:
# Use the estimate and std.error columns from the real data
res = fav_sate.copy()
res['ci_lo'] = res['estimate'] - 1.96 * res['std.error']  # lower bound of 95% CI
res['ci_hi'] = res['estimate'] + 1.96 * res['std.error']  # upper bound of 95% CI

# DerSimonian-Laird random-effects meta-analysis
# Step 1: fixed-effects weights
w_fe = 1.0 / res['std.error']**2  # inverse-variance weights
Q = np.sum(w_fe * (res['estimate'] - np.average(res['estimate'], weights=w_fe))**2)
k = len(res)  # number of studies
# Step 2: estimate between-study variance (tau-squared)
c = w_fe.sum() - (w_fe**2).sum() / w_fe.sum()
tau_sq = max((Q - (k - 1)) / c, 0)  # DL estimator, floored at 0
# Step 3: random-effects weights incorporate tau-squared
w_re = 1.0 / (res['std.error']**2 + tau_sq)
meta_ate = np.average(res['estimate'], weights=w_re)  # pooled estimate
meta_se = np.sqrt(1.0 / w_re.sum())  # pooled SE
meta_p = 2 * (1 - stats.norm.cdf(abs(meta_ate / meta_se)))  # two-sided p-value

print(f"DerSimonian-Laird random-effects meta-analysis:")
print(f"  Pooled ATE: {meta_ate:.4f} (SE = {meta_se:.4f})")
print(f"  95% CI: [{meta_ate - 1.96*meta_se:.4f}, {meta_ate + 1.96*meta_se:.4f}]")
print(f"  p-value: {meta_p:.4f}")
print(f"  tau-squared: {tau_sq:.6f} (between-study variance)")
print(f"  I-squared: {max(0, (Q - (k-1)) / Q * 100):.1f}% (heterogeneity)")

In [ ]:
# Build the forest plot
fig, ax = plt.subplots(figsize=(10, 16))

res_sorted = res.sort_values('estimate').reset_index(drop=True)  # sort by effect size

for i, row in res_sorted.iterrows():
    color = BLUE if row.get('ad_party') == 'Democratic' else RED  # party color
    ax.plot([row['ci_lo'], row['ci_hi']], [i, i],
            color=color, linewidth=1.2, alpha=0.6)  # CI whisker
    ax.plot(row['estimate'], i, 'o',
            color=color, markersize=4)  # point estimate

# Meta-analytic diamond at the bottom (below all studies)
diamond_y = len(res_sorted) + 2
ax.plot(meta_ate, diamond_y, 'D', color='black', markersize=10, zorder=5)
ax.plot([meta_ate - 1.96*meta_se, meta_ate + 1.96*meta_se],
        [diamond_y, diamond_y], 'k-', linewidth=2)  # diamond CI

# Zero-effect reference line
ax.axvline(0, color=GRAY, linestyle='--', alpha=0.5)

ax.set_xlabel('Average Treatment Effect (favorability scale)', fontsize=12)
ax.set_title(
    '59 Ad Experiments: Every Effect Is Small\n'
    '(Blue = pro-Democratic ads, Red = pro-Republican ads, Diamond = meta-analytic average)',
    fontsize=13
)
ax.set_yticks([])  # individual study labels would be too dense
ax.annotate(
    f'Pooled: {meta_ate:.3f}\n(p = {meta_p:.3f})',
    xy=(meta_ate, diamond_y),
    xytext=(0.15, diamond_y - 5),
    fontsize=10,
    arrowprops=dict(arrowstyle='->')
)
plt.tight_layout()
plt.show()

**What you should see:** 59 effects, almost all clustered near zero. Most
confidence intervals cross zero, meaning we cannot reject the null of no
effect for any individual ad. The meta-analytic diamond shows the pooled
average is small (close to zero on the favorability scale). Political ads
just do not move the needle much.

---
## Section 3b: Vote choice ATEs

The favorability outcome measures how warmly respondents feel toward a
candidate. But campaigns care about *votes*. Let us look at the vote-choice
experiments too.

In [ ]:
# Do the same for vote choice SATEs
if 'pro_democrat' in vote_sate.columns:
    vote_sate['ad_party'] = vote_sate['pro_democrat'].map({1: 'Democratic', 0: 'Republican'})
else:
    # Try to merge from ad_meta if needed
    common = set(vote_sate.columns) & set(ad_meta.columns)
    merge_col = list(common - {'estimate', 'std.error', 'conf.low', 'conf.high'})
    if merge_col:
        vote_sate = vote_sate.merge(
            ad_meta[['pro_democrat', merge_col[0]]].drop_duplicates(),
            on=merge_col[0], how='left'
        )
        vote_sate['ad_party'] = vote_sate['pro_democrat'].map({1: 'Democratic', 0: 'Republican'})

print(f"Vote choice experiments: {len(vote_sate)}")
print(f"\nVote choice ATE summary:")
print(vote_sate['estimate'].describe().round(4))

# Plot vote choice forest plot
vote_res = vote_sate.copy()
vote_res['ci_lo'] = vote_res['estimate'] - 1.96 * vote_res['std.error']
vote_res['ci_hi'] = vote_res['estimate'] + 1.96 * vote_res['std.error']

fig, ax = plt.subplots(figsize=(10, 10))
vote_sorted = vote_res.sort_values('estimate').reset_index(drop=True)

for i, row in vote_sorted.iterrows():
    color = BLUE if row.get('ad_party') == 'Democratic' else RED
    ax.plot([row['ci_lo'], row['ci_hi']], [i, i],
            color=color, linewidth=1.2, alpha=0.6)
    ax.plot(row['estimate'], i, 'o', color=color, markersize=5)

ax.axvline(0, color=GRAY, linestyle='--', alpha=0.5)
ax.set_xlabel('Average Treatment Effect (vote choice)', fontsize=12)
ax.set_title(
    f'{len(vote_sate)} Vote Choice Experiments\n'
    '(Blue = pro-Democratic ads, Red = pro-Republican ads)',
    fontsize=13
)
ax.set_yticks([])
plt.tight_layout()
plt.show()

**What you should see:** The vote choice effects are even smaller than the
favorability effects. Ads might nudge how warmly you feel about a candidate,
but actually changing your vote? That is much harder.

---
## Section 4: Subgroup analysis with real CATEs

The CATEs (Conditional Average Treatment Effects) break down each experiment's
effect by respondent characteristics. The key question: do ads work better on
some people than others?

The Coppock data includes CATEs by:
- **pid_3_pre**: partisan identification (Democrat, Republican, Independent)
- **pro_democrat**: whether the ad supports the Democratic candidate

In [ ]:
# Explore the CATE data structure
print("=== Favorability CATEs ===")
print(f"Shape: {fav_cate.shape}")
print(f"Columns: {list(fav_cate.columns)}")
print(fav_cate.head(5))

In [ ]:
# The CATEs should have a subgroup indicator column
# Look for pid_3_pre (partisan ID of the respondent)
if 'pid_3_pre' in fav_cate.columns:
    print("Subgroups in fav_cate by pid_3_pre:")
    print(fav_cate['pid_3_pre'].value_counts())
    subgroup_col = 'pid_3_pre'
else:
    # Find any categorical column that looks like a subgroup
    print("Available columns:", list(fav_cate.columns))
    for col in fav_cate.columns:
        if fav_cate[col].dtype == 'object' or fav_cate[col].nunique() < 10:
            print(f"  {col}: {fav_cate[col].unique()}")

In [ ]:
# Aggregate CATEs by subgroup: average effect for each partisan group
# This answers: do Democratic ads work better on Democrats? On Republicans?

# Get the pro_democrat column for ad party
if 'pro_democrat' not in fav_cate.columns:
    common = set(fav_cate.columns) & set(ad_meta.columns)
    merge_col = list(common - {'estimate', 'std.error', 'conf.low', 'conf.high', 'pid_3_pre'})
    if merge_col:
        fav_cate = fav_cate.merge(
            ad_meta[['pro_democrat', merge_col[0]]].drop_duplicates(),
            on=merge_col[0], how='left'
        )

fav_cate['ad_party'] = fav_cate['pro_democrat'].map({1: 'Democratic', 0: 'Republican'})

# Pool CATEs within each (subgroup x ad_party) cell using inverse-variance weighting
subgroup_results = []
for subgroup in sorted(fav_cate['pid_3_pre'].dropna().unique()):
    for party in ['Democratic', 'Republican']:
        mask = (fav_cate['pid_3_pre'] == subgroup) & (fav_cate['ad_party'] == party)
        sub = fav_cate[mask].dropna(subset=['estimate', 'std.error'])
        if len(sub) == 0:
            continue
        w = 1.0 / sub['std.error']**2  # inverse-variance weights
        pooled_est = np.average(sub['estimate'], weights=w)
        pooled_se = np.sqrt(1.0 / w.sum())
        subgroup_results.append({
            'subgroup': subgroup,
            'ad_party': party,
            'estimate': pooled_est,
            'se': pooled_se,
            'ci_lo': pooled_est - 1.96 * pooled_se,
            'ci_hi': pooled_est + 1.96 * pooled_se,
            'n_experiments': len(sub),
        })

sub_df = pd.DataFrame(subgroup_results)
print("Pooled CATEs by respondent party and ad party:")
print(sub_df.to_string(index=False, float_format='%.4f'))

In [ ]:
# Grouped bar chart: CATEs by respondent partisanship and ad party
fig, ax = plt.subplots(figsize=(10, 6))

subgroups = sorted(sub_df['subgroup'].unique())
x = np.arange(len(subgroups))  # x positions for each subgroup
width = 0.35  # bar width

# Draw one set of bars for Democratic ads, one for Republican ads
for j, (party, color) in enumerate([('Democratic', BLUE), ('Republican', RED)]):
    vals = sub_df[sub_df['ad_party'] == party].set_index('subgroup').reindex(subgroups)
    ax.bar(
        x + j * width - width / 2,  # offset bars side by side
        vals['estimate'],
        width=width,
        yerr=1.96 * vals['se'],  # 95% CI error bars
        color=color,
        alpha=0.8,
        capsize=5,
        label=f'{party} ads'
    )

ax.set_xticks(x)
ax.set_xticklabels(subgroups, fontsize=11)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('Respondent Partisanship', fontsize=12)
ax.set_ylabel('Pooled CATE (favorability)', fontsize=12)
ax.set_title(
    'Do Ads Work Better on Co-Partisans?\n'
    'Conditional Average Treatment Effects by Respondent Party',
    fontsize=13
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**What you should see:** The effects are small across ALL subgroups.
You might expect Democratic ads to help with Democrats and hurt with
Republicans (or vice versa), but the differences are tiny. Partisans
are not dramatically more or less responsive to ads from "their" side.
This is one of the paper's most striking findings: heterogeneity
in ad effects is limited.

In [ ]:
# Same analysis for vote choice CATEs
print("=== Vote Choice CATEs ===")
print(f"Shape: {vote_cate.shape}")
print(f"Columns: {list(vote_cate.columns)}")

if 'pid_3_pre' in vote_cate.columns:
    if 'pro_democrat' not in vote_cate.columns:
        common = set(vote_cate.columns) & set(ad_meta.columns)
        merge_col = list(common - {'estimate', 'std.error', 'conf.low', 'conf.high', 'pid_3_pre'})
        if merge_col:
            vote_cate = vote_cate.merge(
                ad_meta[['pro_democrat', merge_col[0]]].drop_duplicates(),
                on=merge_col[0], how='left'
            )
    vote_cate['ad_party'] = vote_cate['pro_democrat'].map({1: 'Democratic', 0: 'Republican'})

    # Pool vote-choice CATEs by subgroup
    vote_sub_results = []
    for subgroup in sorted(vote_cate['pid_3_pre'].dropna().unique()):
        for party in ['Democratic', 'Republican']:
            mask = (vote_cate['pid_3_pre'] == subgroup) & (vote_cate['ad_party'] == party)
            sub = vote_cate[mask].dropna(subset=['estimate', 'std.error'])
            if len(sub) == 0:
                continue
            w = 1.0 / sub['std.error']**2
            pooled_est = np.average(sub['estimate'], weights=w)
            pooled_se = np.sqrt(1.0 / w.sum())
            vote_sub_results.append({
                'subgroup': subgroup,
                'ad_party': party,
                'estimate': pooled_est,
                'se': pooled_se,
            })

    vote_sub_df = pd.DataFrame(vote_sub_results)
    print("\nPooled vote-choice CATEs:")
    print(vote_sub_df.to_string(index=False, float_format='%.4f'))

---
## Section 5: GOTV simulation (heterogeneous treatment effects)

Coppock et al. found small and homogeneous effects for *persuasion* ads.
But what about *mobilization*? GOTV (Get Out The Vote) campaigns have a
different structure: the treatment effect varies by voter type.

We simulate 10,000 voters to illustrate how treatment effects can vary
even when the overall ATE is modest.

In [ ]:
# Simulate a GOTV experiment with heterogeneous effects
n_v = 10000  # number of simulated voters
voters = pd.DataFrame({
    'age': np.random.uniform(18, 80, n_v),          # uniform age distribution
    'past_votes': np.random.poisson(3, n_v),         # voting history (Poisson)
    'treatment': np.random.binomial(1, 0.5, n_v),   # random assignment (50/50)
})

# True individual treatment effect: larger for young, infrequent voters
# This reflects the Imai & Strauss insight about "persuadables"
voters['true_tau'] = 0.15 * np.exp(-voters['age'] / 40) * np.exp(-voters['past_votes'] / 3)

# Baseline turnout probability: increases with age and past voting
voters['base_prob'] = (0.3 + 0.005 * voters['age'] + 0.08 * voters['past_votes']).clip(0.1, 0.95)

# Realized vote probability = baseline + treatment effect (if treated)
voters['vote_prob'] = (voters['base_prob'] + voters['treatment'] * voters['true_tau']).clip(0.01, 0.99)

# Generate binary outcome
voters['voted'] = np.random.binomial(1, voters['vote_prob'])

# Overall ATE: difference in means between treated and control
ate = voters[voters['treatment'] == 1]['voted'].mean() - voters[voters['treatment'] == 0]['voted'].mean()
print(f"Overall GOTV ATE: {ate:.4f}")

In [ ]:
# Plot heterogeneous effects by age and past voting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in [
    (ax1, 'age', 'Age Group', GREEN),
    (ax2, 'past_votes', 'Past Elections Voted', BLUE)
]:
    bins = pd.qcut(voters[col], 4, duplicates='drop')  # quartiles
    grouped = voters.groupby([bins, 'treatment'])['voted'].mean().unstack()
    grouped['ate'] = grouped[1] - grouped[0]  # ATE within each bin
    ax.bar(range(len(grouped)), grouped['ate'].values, color=color, width=0.5)
    ax.set_xticks(range(len(grouped)))
    ax.set_xticklabels([str(x) for x in grouped.index], fontsize=8, rotation=15)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Treatment Effect', fontsize=11)
    ax.set_title(f'Who Responds to GOTV Contact?\nBy {label}', fontsize=12)
    ax.axhline(0, color='k', linewidth=0.5)

plt.suptitle('Heterogeneous Treatment Effects (Simulated GOTV)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** GOTV effects are largest for young, infrequent
voters. Habitual voters barely respond to contact because they were going
to vote anyway. This is the Imai & Strauss insight: treatment effects are
concentrated among "persuadable" individuals.

---
## Section 6: Causal vs. predictive targeting

Campaigns have limited resources. Who should they contact?

- **Predictive targeting**: contact people least likely to vote (lowest
  baseline probability). This is the intuitive approach.
- **Causal targeting**: contact people most responsive to contact (highest
  individual treatment effect). This is the data-science approach.

These select DIFFERENT people. Let us see which strategy wins.

In [ ]:
n_target = n_v // 10  # budget: can contact 10% of voters

# Predictive targeting: select voters with lowest baseline turnout
pred_idx = voters.nsmallest(n_target, 'base_prob').index

# Causal targeting: select voters with highest true treatment effect
causal_idx = voters.nlargest(n_target, 'true_tau').index

# How much overlap between the two strategies?
overlap = len(set(pred_idx) & set(causal_idx))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: scatter showing which voters each strategy selects
ax1.scatter(voters['base_prob'], voters['true_tau'],
            alpha=0.05, s=5, color=GRAY, label='All voters')
ax1.scatter(voters.loc[pred_idx, 'base_prob'], voters.loc[pred_idx, 'true_tau'],
            alpha=0.3, s=15, color=RED, label='Predictive targets')
ax1.scatter(voters.loc[causal_idx, 'base_prob'], voters.loc[causal_idx, 'true_tau'],
            alpha=0.3, s=15, color=GREEN, label='Causal targets')
ax1.set_xlabel('Baseline Turnout Probability', fontsize=11)
ax1.set_ylabel('Individual Treatment Effect', fontsize=11)
ax1.set_title(f'Two Strategies, Different People\n(Overlap: {overlap/n_target:.0%})', fontsize=12)
ax1.legend(fontsize=10)

# Right panel: total additional votes from each strategy
pred_votes = voters.loc[pred_idx, 'true_tau'].sum()      # expected extra votes
causal_votes = voters.loc[causal_idx, 'true_tau'].sum()   # expected extra votes
ax2.bar(
    ['Predictive\n(unlikely voters)', 'Causal\n(responsive voters)'],
    [pred_votes, causal_votes],
    color=[RED, GREEN],
    width=0.4
)
ax2.set_ylabel('Expected Additional Votes', fontsize=11)
ax2.set_title(f'Causal Targeting: {causal_votes/pred_votes:.1f}x More Votes', fontsize=12)

plt.tight_layout()
plt.show()

**What you should see:** The two strategies select DIFFERENT people.
Predictive targeting picks people who are unlikely to vote (bottom-left
of the scatter), but many of those people are also unresponsive to contact.
Causal targeting picks people who RESPOND to contact (top of the scatter),
producing significantly more additional votes per dollar spent.

---
## Section 7: Connecting back to Coppock et al.

The simulation above illustrates a general principle. But Coppock et al.
found something even more striking: for persuasion ads, there is very
little heterogeneity at all. The CATEs we saw in Section 4 were small
across every subgroup.

This means causal targeting would not help much for persuasion ads,
because there is no subgroup where ads produce large effects. The
practical implication: if your tool only has a small effect on everyone,
no amount of clever targeting will make it powerful.

In [ ]:
# Final comparison: real Coppock CATEs vs. simulated GOTV heterogeneity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of real CATEs from Coppock
ax1.hist(fav_cate['estimate'].dropna(), bins=30, color=BLUE, alpha=0.7, edgecolor='white')
ax1.axvline(0, color='k', linestyle='--', linewidth=0.8)
ax1.axvline(fav_cate['estimate'].mean(), color=RED, linestyle='-', linewidth=2,
            label=f'Mean = {fav_cate["estimate"].mean():.3f}')
ax1.set_xlabel('CATE (favorability)', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title('Real Coppock CATEs:\nLittle Heterogeneity', fontsize=12)
ax1.legend(fontsize=10)

# Right: distribution of simulated individual treatment effects
ax2.hist(voters['true_tau'], bins=30, color=GREEN, alpha=0.7, edgecolor='white')
ax2.axvline(0, color='k', linestyle='--', linewidth=0.8)
ax2.axvline(voters['true_tau'].mean(), color=RED, linestyle='-', linewidth=2,
            label=f'Mean = {voters["true_tau"].mean():.3f}')
ax2.set_xlabel('Individual Treatment Effect', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Simulated GOTV Effects:\nSubstantial Heterogeneity', fontsize=12)
ax2.legend(fontsize=10)

plt.suptitle('When Does Targeting Help?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** The left panel shows real CATEs from Coppock et al.,
tightly clustered around zero with limited spread. The right panel shows
our simulated GOTV effects with a long right tail. Targeting helps when
there IS heterogeneity to exploit (right panel). When effects are uniformly
small (left panel), targeting cannot save you.

---
## Key Takeaways

1. **Ad effects are small.** 59 real experiments, effects near zero on the favorability scale.
2. **Vote choice effects are even smaller.** Changing how people feel is easier than changing how they vote.
3. **Subgroups do not help much.** CATEs show limited heterogeneity across partisan groups.
4. **Treatment effects CAN vary.** In GOTV contexts, young and infrequent voters respond most.
5. **Causal targeting beats predictive targeting** when heterogeneity exists.
6. **But targeting cannot fix a weak treatment.** If the tool does not work on anyone, no targeting strategy will help.